## EDA marathos - silver 
- cleaning


In [0]:
df = spark.sql("FROM marathos.bronze.raw_marathos")
df.display()

In [0]:
df.columns

In [0]:
import re 

#tested to do both functions but will use snake_case for good readibility 

def to_snake_case(name):
    snakecase = snakecase =re.sub(r"[\s]+", "_", name.strip().lower())
    return snakecase

def to_camelCase(name): 
    snakecase = to_snake_case(name)
    words = snakecase.split("_")
    camelcase = words[0].lower() + "".join(word.capitalize() for word in words[1:])
    return camelcase


In [0]:
to_snake_case("Athlete club")

In [0]:
to_camelCase("Athlete club")

In [0]:
def rename_columns_to_snakecase(df): 
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns) 

In [0]:
df_cleaned_columns = rename_columns_to_snakecase(df)
df_cleaned_columns.display()

In [0]:
for column in df.columns: 
    print(column, df.filter(df[column].isNull()).count())

In [0]:
from pyspark.sql.functions import to_timestamp, col, coalesce, lit, when, round as spark_round, lower, trim, regexp_replace, dense_rank, lit, try_to_date

from pyspark.sql.window import Window

In [0]:
df_cleaned_columns.select("event_distance/length", "athlete_performance").limit(30).display()

In [0]:
df_cleaned_columns.select("athlete_performance","athlete_average_speed","event_number_of_finishers").orderBy("athlete_average_speed").display()


In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    when,
    trim,
    regexp_replace,
    try_to_date,
    coalesce,
    round as spark_round,
    regexp_extract,
)


df_cleaned = (
    df_cleaned_columns.withColumn(
        ## Athlete year of birth is converted to an integer, null replaces unknown values
        "athlete_year_of_birth",
        col("athlete_year_of_birth").cast("int"),
    )
    ## Event dates is converted to a date
    .withColumn("event_dates", try_to_date(trim(col("event_dates")), "dd.MM.yyyy"))
    ## Event unit is determined based on the event distance/length column
    .withColumn(
        "event_unit",
        when(col("event_distance/length").contains("km"), "km")
        .when(col("event_distance/length").contains("mi"), "mi")
        .when(col("event_distance/length").contains("h"), "h")
        .otherwise("unknown"),
    )
    ## Performance unit is determined based on the athlete performance column
    .withColumn(
        "performance_unit",
        when(col("athlete_performance").contains("km"), "km")
        .when(col("athlete_performance").contains("h"), "h")
        .otherwise("unknown"),
    )
    ## Event distance/length is cleaned to remove km, mi and h
    .withColumn(
        "event_distance/length",
        regexp_replace(col("event_distance/length"), r"km|mi|h", ""),
    )
    ## Athlete performance is cleaned to remove km, mi and h
    .withColumn(
        "athlete_performance",
        regexp_replace(col("athlete_performance"), r"km|mi|h", ""),
    )
    ## Athlete performance is valid if the athlete performance contains km or h and the event unit is km or mi
    .withColumn(
        "is_valid_performance",
        when(col("athlete_performance").contains("d"), False)
        .when(
            col("event_unit").isin("km", "mi") & (col("performance_unit") == "h"), True
        )
        .when(col("event_unit").isin("h") & (col("performance_unit") == "km"), True)
        .otherwise(False),
    )
    ## Filter and keep the valid performances, then drop the column
    .filter(col("is_valid_performance") == True)
    .drop("is_valid_performance")
    ## Event distance is converted to km as a double
    .withColumn(
        "event_distance_km",
        when(
            col("event_unit") == "km",
            regexp_replace(col("event_distance/length"), "[^0-9.]", "").cast("double"),
        )
        .when(
            col("event_unit") == "mi",
            regexp_replace(col("event_distance/length"), "[^0-9.]", "").cast("double")
            * 1.60934,
        )
        .otherwise(None),
    )
    ## Event distance is converted to hours as a double for timed events
    .withColumn(
        "event_distance_h",
        when(
            col("event_unit") == "h",
            regexp_replace(col("event_distance/length"), "[^0-9.]", "").cast("double"),
        ).otherwise(None),
    )
    ## Athlete performance distance is converted to km as a double
    .withColumn(
        "athlete_performance_distance_km",
        when(
            col("performance_unit") == "km",
            regexp_replace(col("athlete_performance"), "[^0-9.]", "").cast("double"),
        ).otherwise(None),
    )
    ## Athlete performance time is converted to decimal hours as a double
    ## By creating temporary columns for hours, minutes and seconds
    ## and adding them together as decimal hours
    .withColumn(
        "performance_clean",
        when(
            col("performance_unit") == "h", trim(col("athlete_performance"))
        ).otherwise(None),
    )
    .withColumn(
        "_h", regexp_extract(col("performance_clean"), r"^(\d+):", 1).cast("double")
    )
    .withColumn(
        "_m", regexp_extract(col("performance_clean"), r":(\d+):", 1).cast("double")
    )
    .withColumn(
        "_s", regexp_extract(col("performance_clean"), r":(\d+)$", 1).cast("double")
    )
    ## Athlete performance time is converted to decimal hours as a double
    .withColumn(
        "athlete_performance_time_h",
        spark_round(
            when(
                col("performance_unit") == "h",
                col("_h") + (col("_m") / 60) + (col("_s") / 3600),
            ).otherwise(None),
            2,
        ),
    )
    ## Dropping temp columns
    .drop("performance_clean", "_h", "_m", "_s")
    ## calculate avrage speed and replace it into the athlete_average_speed column
    ## Therefor there is a correct calc and less null values
    .withColumn(
        "athlete_average_speed",
        spark_round(
            when(
                col("event_unit").isin("km", "mi"),
                col("event_distance_km") / col("athlete_performance_time_h"),
            )
            .when(
                col("event_unit") == "h",
                col("athlete_performance_distance_km") / col("event_distance_h"),
            )
            .otherwise(None),
            2,
        ),
    )
    ## Event name is cleaned from special characters
    .withColumn(
        "event_name",
        coalesce(
            trim(regexp_replace(col("event_name"), r'^#|"|\*', "")), lit("unknown")
        ),
    )
    ## Athlete club is cleaned from special characters
    .withColumn(
        "athlete_club",
        coalesce(
            trim(regexp_replace(col("athlete_club"), r'^#|"|\*', "")), lit("unknown")
        ),
    )
    .withColumn(
        "athlete_country", coalesce(trim(col("athlete_country")), lit("unknown"))
    )
    .withColumn("athlete_gender", coalesce(trim(col("athlete_gender")), lit("unknown")))
    .withColumn(
        "athlete_age_category",
        coalesce(trim(col("athlete_age_category")), lit("unknown")),
    )
)


df_cleaned.display()

In [0]:
df_cleaned.select(
    "event_distance_km",
    "athlete_performance_time_h",
    "athlete_average_speed"
).show(5)

In [0]:
#this didn't work in streaming table after running pipeline  
#see hashing instead 
# w_event = Window.orderBy("event_name")

# w_race = Window.partitionBy("event_name").orderBy("event_dates")

# df_cleaned = (
#     df_cleaned
    
#     .withColumn("event_id", dense_rank().over(w_event))
    
#     .withColumn("race_id", dense_rank().over(w_race))
# )

In [0]:
df_cleaned.select("athlete_country").distinct().display()

In [0]:
df_countries = spark.sql("FROM marathos.bronze.raw_countries")
df_countries.display()

In [0]:
df_countries.select("country_code_alpha3", "name_long").display()

In [0]:
print("--- DF_CLEANED ---")
df_cleaned.select("athlete_country").distinct().limit(10).show()

print("--- DF_COUNTRIES ---")
df_countries.select("country_code_alpha3").distinct().limit(10).show()

In [0]:

df_countries = df_countries.select("country_code_alpha3", "name_long")


df_cleaned = df_cleaned.join(
    df_countries,
    (df_cleaned["athlete_country"] == df_countries["country_code_alpha3"]) | 
    (df_cleaned["athlete_country"] == df_countries["name_long"]),
    "left"
)

df_cleaned = df_cleaned.withColumn(
    "athlete_country_name",
    when(col("name_long").isNotNull(), col("name_long"))
    .otherwise(col("athlete_country"))
)


df_cleaned = df_cleaned.drop("name_long", "country_code_alpha3")


In [0]:
df_cleaned.display()

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    when,
    trim,
    regexp_replace,
    try_to_date,
    coalesce,
    round as spark_round,
    regexp_extract,
    sha2,
    concat_ws,
)

df_countries = spark.sql("FROM marathos.bronze.raw_countries")

df_countries_athlete = df_countries.select(
    col("country_code_alpha3").alias("athlete_country_code"),
    col("name_long").alias("athlete_country_long"),
)

df_countries_event = df_countries.select(
    col("country_code_alpha3").alias("event_country_code"),
    col("name_long").alias("event_country_long"),
)

df_cleaned = (
    df_cleaned_columns.join(
        df_countries_athlete,
        (
            df_cleaned_columns["athlete_country"]
            == df_countries_athlete["athlete_country_code"]
        )
        | (
            df_cleaned_columns["athlete_country"]
            == df_countries_athlete["athlete_country_long"]
        ),
        "left",
    )
    #### EVENT ############
    ## Event name is cleaned from special characters
    .withColumn(
        "event_name",
        coalesce(
            trim(regexp_replace(col("event_name"), r'^#|"|\*', "")), lit("unknown")
        ),
    )
    ## Event dates is converted to a date
    .withColumn("event_dates", try_to_date(trim(col("event_dates")), "dd.MM.yyyy"))
    ## Event country is extracted from the event name
    .withColumn(
        "event_country",
        when(
            regexp_extract(col("event_name"), r"\(([A-Z]{3})\)", 1) == "", None
        ).otherwise(regexp_extract(col("event_name"), r"\(([A-Z]{3})\)", 1)),
    )
    ## Event unit is determined based on the event distance/length column
    .withColumn(
        "event_unit",
        when(col("event_distance/length").contains("km"), "km")
        .when(col("event_distance/length").contains("mi"), "mi")
        .when(col("event_distance/length").contains("h"), "h")
        .otherwise("unknown"),
    )
    ## Event distance/length is cleaned to remove km, mi and h
    .withColumn(
        "event_distance/length",
        regexp_replace(col("event_distance/length"), r"km|mi|h", ""),
    )
    ## Event distance is converted to km as a double
    .withColumn(
        "event_distance_km",
        when(
            col("event_unit") == "km",
            regexp_replace(col("event_distance/length"), "[^0-9.]", "").cast("double"),
        )
        .when(
            col("event_unit") == "mi",
            regexp_replace(col("event_distance/length"), "[^0-9.]", "").cast("double")
            * 1.60934,
        )
        .otherwise(None),
    )
    ## Event distance is converted to hours as a double for timed events
    .withColumn(
        "event_distance_h",
        when(
            col("event_unit") == "h",
            regexp_replace(col("event_distance/length"), "[^0-9.]", "").cast("double"),
        ).otherwise(None),
    )
    ## Event ID is generated from the event name
    .withColumn(
        "event_id",
        sha2(col("event_name"), 256),
    )
    ## Race ID is generated from the event name and date
    .withColumn(
        "race_id",
        sha2(concat_ws("_", col("event_name"), col("event_dates")), 256),
    )
    #### ATHLETE ############
    ## Athlete year of birth is converted to an integer, null replaces unknown values
    .withColumn(
        "athlete_year_of_birth",
        col("athlete_year_of_birth").cast("int"),
    )
    ## Athlete club is cleaned from special characters
    .withColumn(
        "athlete_club",
        coalesce(
            trim(regexp_replace(col("athlete_club"), r'^#|"|\*', "")), lit("unknown")
        ),
    )
    .withColumn(
        "athlete_country", coalesce(trim(col("athlete_country")), lit("unknown"))
    )
    ## Athlete country name is mapped from the country code
    .withColumn(
        "athlete_country_name",
        when(
            col("athlete_country_long").isNotNull(), col("athlete_country_long")
        ).otherwise(col("athlete_country")),
    )
    .withColumn("athlete_gender", coalesce(trim(col("athlete_gender")), lit("unknown")))
    .withColumn(
        "athlete_age_category",
        coalesce(trim(col("athlete_age_category")), lit("unknown")),
    )
    .drop("athlete_country_code", "athlete_country_long")
    #### PERFORMANCE ############
    ## Performance unit is determined based on the athlete performance column
    .withColumn(
        "performance_unit",
        when(col("athlete_performance").contains("km"), "km")
        .when(col("athlete_performance").contains("h"), "h")
        .otherwise("unknown"),
    )
    ## Athlete performance is cleaned to remove km, mi and h
    .withColumn(
        "athlete_performance",
        regexp_replace(col("athlete_performance"), r"km|mi|h", ""),
    )
    ## Athlete performance is valid if the athlete performance contains km or h and the event unit is km or mi
    .withColumn(
        "is_valid_performance",
        when(col("athlete_performance").contains("d"), False)
        .when(
            col("event_unit").isin("km", "mi") & (col("performance_unit") == "h"), True
        )
        .when(col("event_unit").isin("h") & (col("performance_unit") == "km"), True)
        .otherwise(False),
    )
    ## Filter and keep the valid performances, then drop the column
    .filter(col("is_valid_performance") == True)
    .drop("is_valid_performance")
    ## Athlete performance distance is converted to km as a double
    .withColumn(
        "athlete_performance_distance_km",
        when(
            col("performance_unit") == "km",
            regexp_replace(col("athlete_performance"), "[^0-9.]", "").cast("double"),
        ).otherwise(None),
    )
    ## Athlete performance time is converted to decimal hours as a double
    ## By creating temporary columns for hours, minutes and seconds
    ## and adding them together as decimal hours
    .withColumn(
        "performance_clean",
        when(
            col("performance_unit") == "h", trim(col("athlete_performance"))
        ).otherwise(None),
    )
    .withColumn(
        "_h", regexp_extract(col("performance_clean"), r"^(\d+):", 1).cast("double")
    )
    .withColumn(
        "_m", regexp_extract(col("performance_clean"), r":(\d+):", 1).cast("double")
    )
    .withColumn(
        "_s", regexp_extract(col("performance_clean"), r":(\d+)$", 1).cast("double")
    )
    ## Athlete performance time is converted to decimal hours as a double
    .withColumn(
        "athlete_performance_time_h",
        spark_round(
            when(
                col("performance_unit") == "h",
                col("_h") + (col("_m") / 60) + (col("_s") / 3600),
            ).otherwise(None),
            2,
        ),
    )
    ## Dropping temp columns
    .drop("performance_clean", "_h", "_m", "_s")
    ## calculate avrage speed and replace it into the athlete_average_speed column
    ## Therefor there is a correct calc and less null values
    .withColumn(
        "athlete_average_speed",
        spark_round(
            when(
                col("event_unit").isin("km", "mi"),
                col("event_distance_km") / col("athlete_performance_time_h"),
            )
            .when(
                col("event_unit") == "h",
                col("athlete_performance_distance_km") / col("event_distance_h"),
            )
            .otherwise(None),
            2,
        ),
    )
    ## adding result id from race id and athlete id 
    .withColumn(
        "result_id",
        sha2(concat_ws("_", col("race_id"), col("athlete_id").cast("string")), 256),
    )
)

## Event country name is mapped from the country code
df_cleaned = df_cleaned.join(
    df_countries_event,
    df_cleaned["event_country"] == df_countries_event["event_country_code"],
    "left",
)

df_cleaned = df_cleaned.withColumn(
    "event_country_name",
    when(col("event_country_long").isNotNull(), col("event_country_long")).otherwise(
        col("event_country")
    ),
).drop("event_country_code", "event_country_long")

df_cleaned.display()

In [0]:
## source for this is in databricks documentation and apache pyspark docs
from pyspark.sql.functions import sha2, concat_ws 

df_cleaned = df_cleaned.withColumn(
    "event_id", 
    sha2(col("event_name"), 256)
).withColumn(
    "race_id", 
    sha2(concat_ws("_", col("event_name"), col("event_dates"), col("event_country")),256)
)


df_cleaned.select("event_id", "race_id").display()


In [0]:
from pyspark.sql.functions import concat_ws, sha2, col

df_cleaned.select(
    col("race_id"),
    col("athlete_id"),
    concat_ws("_", col("race_id"), col("athlete_id").cast("string")).alias("concat_result"),
    sha2(concat_ws("_", col("race_id"), col("athlete_id").cast("string")), 256).alias("result_id_test")
).show(5)

In [0]:
df_cleaned.select("athlete_id").display()